In [3]:
%load_ext autoreload
%autoreload 2

# `Logit` on Orders - Logistic Regression (~1h)

## Select features

🎯 Haydi `wait_time` ve `delay_vs_expected` değişkenlerinin çok `iyi/kötü review`lar üzerindeki etkisini inceleyelim.

👉 `orders` training_set’imizi kullanarak iki adet `multivariate logistic regression` çalıştıracağız:
- `logit_one` → `dim_is_one_star` tahmini için  
- `logit_five` → `dim_is_five_star` tahmini için.

 

In [4]:
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

👉 Dataset’inizi import edin:

In [5]:
from olist.order import Order
orders = Order().get_training_data(with_distance_seller_customer=True)

👉 Kullanmak istediğiniz feature’ları bir listede seçin:

⚠️ Data leakage yaratmadığınızdan emin olun (yani target’tan türetilmiş feature’ları seçmeyin)

💡 `wait_time` ve `delay_vs_expected` değişkenlerinin etkisini anlayabilmek için diğer feature’ların etkisini kontrol etmemiz gerekir, bu yüzden listenize ilgili olabilecek tüm feature’ları dahil edin.

In [6]:
features = [
    'wait_time',
    'delay_vs_expected',
    'dim_is_five_star',
    'dim_is_one_star',
    'number_of_products',
    'number_of_sellers',
    'price',
    'freight_value',
    'distance_seller_customer'
]

# Hedef değişkenler dışındaki bağımsız değişkenler (özellikler listesi)
features_to_model = [
    'wait_time',
    'delay_vs_expected',
    'number_of_products',
    'number_of_sellers',
    'price',
    'freight_value',
    'distance_seller_customer'
]

df = orders[features].dropna()
df.head()

,wait_time,delay_vs_expected,dim_is_five_star,dim_is_one_star,number_of_products,number_of_sellers,price,freight_value,distance_seller_customer
0,8.436574,0.0,0,0,1,1,29.99,8.72,18.576110
1,13.782037,0.0,0,0,1,1,118.70,22.76,851.495069
2,9.394213,0.0,1,0,1,1,159.90,19.22,514.410666
3,13.208750,0.0,1,0,1,1,45.00,27.20,1822.226336
4,2.873877,0.0,1,0,1,1,19.90,8.72,29.676625


🕵🏻 Feature’larınızın `multicollinearity` durumunu `VIF index` kullanarak kontrol edin.

* Çok yüksek olmamalıdır (tercihen < 10), böylece partial regression coefficient’larına ve ilgili `p-values` değerlerine güvenebiliriz.
* Verinizi standardize etmeyi unutmayın!
    * Bir `VIF Analysis`, bir feature’ın diğer feature’lara karşı regresyonunu yaparak hesaplanır...
    * Bu yüzden herhangi bir linear regression çalıştırmadan önce feature’ların `scale etkisini kaldırmak` ve eşit öneme sahip olmalarını sağlamak istersiniz!
    
    
📚 <a href="https://www.statisticshowto.com/variance-inflation-factor/">Statistics How To - Variance Inflation Factor</a>

📚  <a href="https://online.stat.psu.edu/stat462/node/180/">PennState - Detecting Multicollinearity Using Variance Inflation Factors</a>

⚖️ Standardize etme:

In [7]:
df_standardized = df.copy()

for feature in features_to_model:
    mu = df[feature].mean()
    sigma = df[feature].std()
    df_standardized[feature] = (df[feature] - mu) / sigma

df_standardized.head()

,wait_time,delay_vs_expected,dim_is_five_star,dim_is_one_star,number_of_products,number_of_sellers,price,freight_value,distance_seller_customer
0,-0.430413,-0.161672,0,0,-0.264492,-0.112242,-0.513662,-0.651492,-0.981091
1,0.134759,-0.161672,0,0,-0.264492,-0.112242,-0.087204,0.000191,0.423276
2,-0.329163,-0.161672,1,0,-0.264492,-0.112242,0.110857,-0.164122,-0.145075
3,0.074146,-0.161672,1,0,-0.264492,-0.112242,-0.441504,0.206278,2.060007
4,-1.018554,-0.161672,1,0,-0.264492,-0.112242,-0.562168,-0.651492,-0.962375


👉 Olası multicollinearity durumlarını analiz etmek için VIF Analysis’inizi çalıştırın:

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data["feature"] = features_to_model
vif_data["VIF"] = [variance_inflation_factor(df_standardized[features_to_model].values, i)
                   for i in range(len(features_to_model))]

display(vif_data.sort_values(by="VIF", ascending=False))

,feature,VIF
0,wait_time,2.630920
1,delay_vs_expected,2.217007
5,freight_value,1.673621
6,distance_seller_customer,1.445789
2,number_of_products,1.369862
4,price,1.208175
3,number_of_sellers,1.092612


## Logistic Regressions

👉 İki adet `Logistic Regression` modeli fit edin:
- `logit_one` → `dim_is_one_star` tahmini için
- `logit_five` → `dim_is_five_star` tahmini için.

`Logit 1️⃣`

In [9]:
formula_one = f"dim_is_one_star ~ {' + '.join(features_to_model)}"
logit_one = smf.logit(formula=formula_one, data=df_standardized).fit()
logit_one.summary()

Optimization terminated successfully.
         Current function value: 0.273327
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:        dim_is_one_star   No. Observations:                95349
Model:                          Logit   Df Residuals:                    95341
Method:                           MLE   Df Model:                            7
Date:                Sat, 31 Oct 2026   Pseudo R-squ.:                  0.1450
Time:                        16:14:04   Log-Likelihood:                -26061.
converged:                       True   LL-Null:                       -30481.
Covariance Type:            nonrobust   LLR p-value:                     0.000
============================================================================================
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                   -2.4690      0.013   -190.309      0.000      -2.494      -2.444
wait_time                    0.7135      0.017     42.057      0.000       0.680       0.747
delay_vs_expected            0.2560      0.019     13.761      0.000       0.220       0.292
number_of_products           0.2487      0.011     23.455      0.000       0.228       0.269
number_of_sellers            0.1786      0.008     22.694      0.000       0.163       0.194
price                        0.0511      0.011      4.516      0.000       0.029       0.073
freight_value               -0.0223      0.013     -1.719      0.086      -0.048       0.003
distance_seller_customer    -0.1782      0.014    -12.859      0.000      -0.205      -0.151
============================================================================================
"""

`Logit 5️⃣`

In [10]:
formula_five = f"dim_is_five_star ~ {' + '.join(features_to_model)}"
logit_five = smf.logit(formula=formula_five, data=df_standardized).fit()
logit_five.summary()

Optimization terminated successfully.
         Current function value: 0.636783
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:       dim_is_five_star   No. Observations:                95349
Model:                          Logit   Df Residuals:                    95341
Method:                           MLE   Df Model:                            7
Date:                Sat, 31 Oct 2026   Pseudo R-squ.:                 0.05807
Time:                        16:14:14   Log-Likelihood:                -60717.
converged:                       True   LL-Null:                       -64460.
Covariance Type:            nonrobust   LLR p-value:                     0.000
============================================================================================
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                    0.3394      0.007     47.309      0.000       0.325       0.353
wait_time                   -0.5236      0.012    -44.659      0.000      -0.547      -0.501
delay_vs_expected           -0.4305      0.024    -18.261      0.000      -0.477      -0.384
number_of_products          -0.1355      0.008    -16.184      0.000      -0.152      -0.119
number_of_sellers           -0.1432      0.008    -18.215      0.000      -0.159      -0.128
price                        0.0207      0.008      2.700      0.007       0.006       0.036
freight_value                0.0050      0.009      0.559      0.576      -0.013       0.023
distance_seller_customer     0.0878      0.008     10.542      0.000       0.071       0.104
============================================================================================
"""

💡 Şimdi bu iki logistic regression’ın sonuçlarını analiz etme zamanı:

- Partial coefficient’ları kendi kelimelerinizle yorumlayın.
- `p-values` kullanarak istatistiksel anlamlılıklarını kontrol edin.
- Coefficient önemleri açısından `logit_one` ve `logit_five` arasında herhangi bir fark görüyor musunuz?

In [13]:
# Aşağıdaki cümlelerden doğru olanları aşağıdaki listeye kaydedin.

a = "delay_vs_expected influences five_star ratings even more than one_star ratings"
b = "wait_time influences five_star ratings even more than one_star"

your_answer = [a]

🧪 __Kodunu Test Et__

In [14]:
from nbresult import ChallengeResult

result = ChallengeResult('logit',
    answers = your_answer
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/ozgek/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/ozgek/code/ozge-karadeniiz/data-logit/tests
plugins: typeguard-4.4.2, anyio-4.8.0
collecting ... collected 1 item

test_logit.py::TestLogit::test_question PASSED                           [100%]

============================== 1 passed in 0.06s ===============================


💯 You can commit your code:

git add tests/logit.pickle

git commit -m 'Completed logit step'

git push origin master



<details>
    <summary>- <i>Açıklamalar ve ileri seviye kavramlar</i> -</summary>


> _Diğer tüm şeyler sabitken, `delay factor`, 1-yıldız review alma ihtimalini etkilemesinden bile daha fazla, 5-yıldızdan mahrum kalma ihtimalini artırma eğilimindedir. Muhtemelen bunun sebebi, 1-yıldız review’ların bizzat çok kötü ürünleri hedeflemesi, kötü teslimatları değil._

❗️ Ancak tamamen titiz olmak için, **iki farklı modelin coefficient’larını karşılaştırırken daha dikkatli olmamız gerekir**, çünkü **benzer popülasyonlara dayanmayabilirler**!
    Burada 2 alt popülasyonumuz var: (1-yıldız verenler ve 5-yıldız verenler) ve bunlar doğaları gereği farklı davranış kalıpları sergileyebilirler. 5-yıldız vermeye daha meyilli “mutlu insanlar”ın, “gecikme” veya “fiyat” söz konusu olduğunda, 1-yıldızı “Lucky-Luke gibi ateşleyen” “huysuz insanlara” göre daha az hassas olmaları gayet mümkün...

</details>



🏁 Tebrikler!

💾 `logit.ipynb` notebook’unuzu commit ve push etmeyi unutmayın!